In [13]:
import csv
import importlib
import os
import random
import sys
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from time import sleep
from collections import deque, defaultdict
from itertools import count
from typing import Any, Dict, Counter, List

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

from importnb import Notebook
with Notebook():
    from Labs.LatencyModel import LatencyModel, MultiDULatencyModel
    from Labs.Policy import DrlPolicy
    from Labs.CacheEngine import CacheEngineEnv
    from Labs.UserRequest import UserRequestEvents
    from Labs.EnvWrapper import EnvWrapper

from RL.Networks import QNetwork, MultiHeadQNetwork
from RL.Buffers import ReplayBuffer, NStepReplayBuffer
from RL.Adapters import FeatureAdapter, NetworkAdapter
import RL.PPOWorkers as PPOWorkers

import Common.config as config
import Common.datatypes as datatypes
import Common.debugger as debugger
import Common.utils as utils
import Core.builders as builders

importlib.reload(builders)
importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(debugger)
importlib.reload(utils)
importlib.reload(PPOWorkers)

BaseWorker = PPOWorkers.BaseWorker
EnhWorker = PPOWorkers.EnhWorker

In [14]:
cfg = config.Config()
cfg.filename = \
    f"focus_eps{cfg.epsilon_start}_" \
    f"lrdecay{cfg.learning_rate_decay}_" \
    f"gamma{cfg.gamma}.csv"
cfg.hidden_dims = [128, 128]
cfg.K_epochs = 4
cfg.gae_lambda = 0.99
cfg.coef_entropy = 0.01
cfg.clip_ratio = 0.2
cfg.value_loss_coef = 0.5
cfg.nb_interval = 1000

debugger = debugger.debug

In [15]:
def save_training_results(
    path_,
    filename,
    ep, 
    total_reward, 
    cache_hits, 
    cache_misses, 
    agent
):
    with open(os.path.join(path_, filename), 'a', newline='') as f:
        fieldnames = [
            'episode', 
            'total_reward', 
            'cache_hits', 
            'cache_misses'
        ]
        writer_results = csv.DictWriter(f, fieldnames=fieldnames)

        if ep == 0:
            writer_results.writeheader()


        writer_results.writerow({
            'episode': ep,
            'total_reward': round(float(total_reward), 2),
            'cache_hits': cache_hits,
            'cache_misses': cache_misses
        })

def update_metrics(info: dict, reward: float) -> tuple[float, int, int, int, int]:
    enh_hits = info.get("enh_layer_hits", 0)
    base_hits = info.get("base_layer_hits", 0)
    enh_misses = info.get("enh_layer_misses", 0)
    base_misses = info.get("base_layer_misses", 0)

    return reward, base_hits, base_misses, enh_hits, enh_misses


In [16]:
def select_action(state, base_agent, enh_agent):
    state_base, state_enh = state
    base_action, base_log_prob, base_value = base_agent.select_action(state_base)
    enh_action, enh_log_prob, enh_value = enh_agent.select_action(state_enh)

    joint_action = np.concatenate([np.array([base_action]), np.array(enh_action)], axis=0)
    return joint_action, base_log_prob, base_value, enh_log_prob, enh_value

def run_episode(episode, env, base_agent, enh_agent, net_adapter, cfg):
    """Run one full training episode."""
    _, info = net_adapter.reset()

    total_reward = 0.0
    cache_hits = cache_misses = 0
    base_hits = base_misses = 0
    enh_hits = enh_misses = 0

    env.warmup_phase(net_adapter)

    for step in count():

        # --- Build State ---
        req_state = info.get("user_request", None)
        state = net_adapter.build_observation(req_state)

        # --- Action Selection --- 
        action, base_log_prob, base_value, enh_log_prob, enh_value = select_action(state, base_agent, enh_agent)

        # --- Environment Step ---
        _, reward, done, info = env.step(action, req_state, net_adapter)

        # --- Store Transition & Train ---
        reward_0 = info["reward_layer_0"]
        reward_1 = info["reward_layer_1"]
        prefetch_base = info["prefetch_base"]
        prefetch_enh = info["prefetch_enh"]

        state_base, state_enh = state
        if prefetch_base:
            base_agent.remember(
                state_base, int(action[0]), base_log_prob, base_value, reward_0, done
            )
            base_agent.train_step()

        if prefetch_enh:
            enh_agent.remember(
                state_enh, action[1:], enh_log_prob, enh_value, reward_1, done
            )
            enh_agent.train_step()

        delta_r, bs_hits, bs_miss, e_hits, e_miss = update_metrics(info, reward)
        total_reward += delta_r
        cache_hits += bs_hits + e_hits
        cache_misses += bs_miss + e_miss
        base_hits += bs_hits
        base_misses += bs_miss
        enh_hits += e_hits
        enh_misses += e_miss

        if done:
            break

        debugger.log('cache_hits', bs_hits + e_hits)
        debugger.log('cache_misses', bs_miss + e_miss)

    return total_reward, cache_hits, cache_misses, base_hits, base_misses, enh_hits, enh_misses

def train(cfg):
    env = builders.build_environment(cfg)

    base_agent = BaseWorker(cfg, debugger=debugger)
    enh_agent = EnhWorker(cfg, debugger=debugger)

    feature_adapter = FeatureAdapter(cfg, env)
    net_adapter = NetworkAdapter(cfg, env, feature_adapter)

    date_dir = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M")
    debug_path = os.path.join(cfg.path_results, date_dir)
    os.makedirs(debug_path, exist_ok=True)

    print(f"Starting training for {cfg.n_episodes} episodes... {date_dir}")

    for episode in range(cfg.n_episodes):

        total_reward, hits, misses, bs_hits, bs_miss, enh_hits, enh_miss = run_episode(
            episode, env, base_agent, enh_agent, net_adapter, cfg
        )

        save_training_results(
            path_=cfg.path_results,
            filename=cfg.filename,
            ep=episode,
            total_reward=total_reward,
            cache_hits=hits,
            cache_misses=misses,
            agent=enh_agent
        )

        print(
            f"--- Episode {episode} | R: {total_reward:.2f} | "
            f"HR: {hits / (hits + misses + 1e-9):.2f} | "
            f"BHR: {bs_hits / (bs_hits + bs_miss + 1e-9):.2f} | "
            f"EHR: {enh_hits / (enh_hits + enh_miss + 1e-9):.2f} ---"
        )

        debugger.save_results(filepath=f"{debug_path}/debug_ep{episode}")
        debugger.clear()

        print("-" * 50)

if __name__ == "__main__":
    train(cfg)

NetworkAdapter initialized with capacity: 50 videos, 4 tiles per video
Starting training for 300 episodes... 2026-03-02_19-58
--- Episode 0 | R: 3997.70 | HR: 0.35 | BHR: 0.43 | EHR: 0.09 ---
--------------------------------------------------
--- Episode 1 | R: 1100.40 | HR: 0.26 | BHR: 0.33 | EHR: 0.04 ---
--------------------------------------------------
--- Episode 2 | R: 1519.68 | HR: 0.29 | BHR: 0.38 | EHR: 0.05 ---
--------------------------------------------------
--- Episode 3 | R: 1286.21 | HR: 0.28 | BHR: 0.36 | EHR: 0.04 ---
--------------------------------------------------
--- Episode 4 | R: 2225.10 | HR: 0.33 | BHR: 0.42 | EHR: 0.06 ---
--------------------------------------------------
--- Episode 5 | R: 1462.68 | HR: 0.28 | BHR: 0.36 | EHR: 0.05 ---
--------------------------------------------------
--- Episode 6 | R: 1944.48 | HR: 0.30 | BHR: 0.39 | EHR: 0.05 ---
--------------------------------------------------
--- Episode 7 | R: 1469.99 | HR: 0.29 | BHR: 0.38 | EHR

KeyboardInterrupt: 